In [79]:
import yaml
from types import SimpleNamespace
# from strategies.run_allen import get_allen_result, get_allen_signals
from strategies.run_gino import get_gino_result, get_gino_signals
from strategies.utils.analysis import get_equal_weight_baseline_result
from utils.io import read_all_df
import pandas as pd
from evaluation.stats import sharpe
import finlab
from finlab import data

finlab.login('ntSS3778pZi2FfkeYxXP0p+S0iI4AggkcphAUxh/lTVrWqT2FreKQsDkTA92CM7d#vip_m')

CATEGORY = 'Selected'

result_dir = f"results/FFT/{CATEGORY}_Dataset_Abs"


def load_config(path):
    with open(path, "r") as f:
        cfg_dict = yaml.safe_load(f)
    return SimpleNamespace(**cfg_dict)
backtest_config = load_config("config/backtest.yaml")

f = open("config/backtest.yaml")
cfg = yaml.safe_load(f)

result = get_gino_result(cfg, result_dir)
signals = get_gino_signals(cfg, result_dir)
test_pred, test_truth, _, _ = read_all_df(result_dir)

輸入成功!
Exceeded maximum positions of 30 at 2021-02-19 00:00:00 for symbol 2344!
Exceeded maximum positions of 30 at 2021-02-22 00:00:00 for symbol 2338!
Exceeded maximum positions of 30 at 2021-02-23 00:00:00 for symbol 2609!
Exceeded maximum positions of 30 at 2021-02-24 00:00:00 for symbol 2609!
Exceeded maximum positions of 30 at 2021-02-25 00:00:00 for symbol 2609!
Exceeded maximum positions of 30 at 2021-02-26 00:00:00 for symbol 2609!
Exceeded maximum positions of 30 at 2021-03-02 00:00:00 for symbol 2609!
Exceeded maximum positions of 30 at 2021-03-03 00:00:00 for symbol 2609!
Not Enough Money to buy at 2021-10-01 00:00:00 for stock 2486!
Not Enough Money to buy at 2021-10-15 00:00:00 for stock 3504!
Not Enough Money to buy at 2021-11-26 00:00:00 for stock 3016!
Not Enough Money to buy at 2022-01-17 00:00:00 for stock 2359!
Not Enough Money to buy at 2022-04-11 00:00:00 for stock 3035!
Not Enough Money to buy at 2022-06-10 00:00:00 for stock 8996!
Not Enough Money to buy at 2022-

In [80]:
baseline_results = get_equal_weight_baseline_result(result_dir, pd.to_datetime('2021-01-01'), pd.to_datetime('2025-09-21'))

In [81]:
def yearly_returns(returns):
    yearly_roi = returns.resample("Y").last() / returns.resample("Y").first() - 1
    rois = {}
    for year in yearly_roi.index:
        rois[year.year] = yearly_roi[year]
    rois['final'] = returns.iloc[-1] / returns.iloc[0] - 1
    rois['sharpe'] = sharpe(returns)
    return rois

In [82]:
model_rois = yearly_returns(result['model'].returns)
baseline_rois = yearly_returns(baseline_results.returns)

print("model rois =", model_rois)
print("baseline rois =", baseline_rois)

model rois = {2021: 0.46639684752509813, 2022: -0.1973552989627425, 2023: 1.3290065243337583, 2024: 1.07972791909951, 2025: 0.21429446441283395, 'final': 5.803327489621953, 'sharpe': 1.1962765981422}
baseline rois = {2021: 0.5255148315959293, 2022: -0.177160919352212, 2023: 0.5422643296083989, 2024: 0.1678033943383097, 2025: 0.11786361266109169, 'final': 1.502652278390101, 'sharpe': 0.9832177487976348}


In [ ]:
from evaluation.surge_metrics import compute_surge_metrics_from_buy_signals

model_data = {
    'category': CATEGORY,
    'baseline': 0,
}
model_data.update(model_rois)

baseline_data = {
    'category': CATEGORY,
    'baseline': 1,
}
baseline_data.update(baseline_rois)

result_df = pd.DataFrame([model_data, baseline_data])
result_df


,category,baseline,2021,2022,2023,2024,2025,final,sharpe
0,Selected,0,0.466397,-0.197355,1.329007,1.079728,0.214294,5.803327,1.196277
1,Selected,1,0.525515,-0.177161,0.542264,0.167803,0.117864,1.502652,0.983218


In [84]:
model_df = pd.DataFrame([model_data])
baseline_df = pd.DataFrame([baseline_data])

display_df = pd.DataFrame(index=model_df.index)
display_df['category'] = model_df['category']

for col in model_df.columns:
    if col == 'category':
        continue
    model_values = model_df[col]
    baseline_values = baseline_df[col]
    delta = model_values - baseline_values
    if str(col) in ['2021', '2022', '2023', '2024', '2025', 'final']:
        display_df[col] = model_values.mul(100).round(1).apply(lambda x: f"{x:.0f}%") + \
                        delta.mul(100).round(1).apply(lambda x: f"({x:+.0f}%)")

    else:
        display_df[col] = model_values.round(2).astype(str) + \
                            delta.round(2).apply(lambda x: f" ({x:+.2f})")
                            
display_df = display_df.drop(columns=['baseline'])
display_df

,category,2021,2022,2023,2024,2025,final,sharpe
0,Selected,47%(-6%),-20%(-2%),133%(+79%),108%(+91%),21%(+10%),580%(+430%),1.2 (+0.21)
